# **PHẦN 4.2: PHÁT HIỆN BẤT THƯỜNG**
## **1. Định nghĩa vấn đề**
+ **Mô tả**:
   - Phát hiện các bất thường nhằm phân biệt tốt giữa lớp bình thường (Benign) và lớp bất thường (Android_Adware Android_Scareware, Android_SMS_Malware)

## **2. Chuẩn bị vấn đề**

### 2.1. Import các thư viện cần thiết

In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow
import math
import itertools

from imblearn.over_sampling import SMOTENC

from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EllipticEnvelope

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense


from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score, auc, roc_curve, confusion_matrix
from sklearn.preprocessing import label_binarize

import time
import joblib

RANDOM_STATE = 42

pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 500)

### 2.2. Lấy tập dữ liệu đã xử lý

In [52]:
x_train = pd.read_parquet("../data_processed/x_train.parquet")
x_train.head()

,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Bwd Packet Length Max,Bwd Packet Length Min,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Bwd IAT Total,Bwd IAT Mean,Fwd PSH Flags,Fwd Header Length,Bwd Packets/s,Min Packet Length,FIN Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,Down/Up Ratio,Init_Win_bytes_backward,Active Mean,Active Std,Idle Mean,Idle Std,s_bit_0,s_bit_1,s_bit_2,s_bit_3,s_bit_4,s_bit_5,s_bit_6,s_bit_7,s_bit_8,s_bit_9,s_bit_10,s_bit_11,s_bit_12,s_bit_13,s_bit_14,s_bit_15,s_bit_16,s_bit_17,s_bit_18,s_bit_19,s_bit_20,s_bit_21,s_bit_22,s_bit_23,s_bit_24,s_bit_25,s_bit_26,s_bit_27,s_bit_28,s_bit_29,s_bit_30,s_bit_31,d_bit_0,d_bit_1,d_bit_2,d_bit_3,d_bit_4,d_bit_5,d_bit_6,d_bit_7,d_bit_8,d_bit_9,d_bit_10,d_bit_11,d_bit_12,d_bit_13,d_bit_14,d_bit_15,d_bit_16,d_bit_17,d_bit_18,d_bit_19,d_bit_20,d_bit_21,d_bit_22,d_bit_23,d_bit_24,d_bit_25,d_bit_26,d_bit_27,d_bit_28,d_bit_29,d_bit_30,d_bit_31,Source Port_Grp_System,Source Port_Grp_Registered,Source Port_Grp_Dynamic,Destination Port_Grp_System,Destination Port_Grp_Registered,Destination Port_Grp_Dynamic,Destination Port_Flag_HTTP,Destination Port_Flag_HTTPS,Destination Port_Flag_DNS,Destination Port_Flag_Reserved,Destination Port_Flag_SSDP,Destination Port_Flag_Other_System,Destination Port_Flag_Other_Registered,Destination Port_Flag_Dynamic,Hour_sin,Hour_cos,Day_sin,Day_cos,Protocol_0.0,Protocol_6.0,Protocol_17.0,DayOfWeek_0,DayOfWeek_1,DayOfWeek_2,DayOfWeek_3,DayOfWeek_4,DayOfWeek_5,Init_Win_bytes_Bwd_Missing
341993,-1.756805,0.000000,-0.570892,-0.592211,0.0,-0.601360,0.0,-0.580821,2.341243,-1.924641,0.000000,0.000000,0.000000,0.0,-0.279107,-0.220216,0.0,0.0,0.0,1.0,0.0,0,0.000000,0.000000,0.0,0.000000,0.0,1,1,0,0,1,0,1,0,0,1,0,0,1,1,0,1,1,0,0,0,0,0,0,1,1,0,0,1,0,1,1,0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,1,0,5.000000e-01,-0.866025,0.848644,0.528964,0,1,0,0,0,0,0,1,0,1
300839,0.866255,0.261860,0.000000,0.000000,0.0,-0.601360,0.0,-0.527324,-0.573250,1.000553,1.218707,0.000000,0.000000,0.0,0.325849,-0.214466,0.0,0.0,0.0,1.0,1.0,0,0.000000,11.407432,0.0,17.908537,0.0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,1,1,1,1,0,1,0,1,1,0,0,1,1,0,1,1,0,0,1,0,0,0,0,1,1,0,0,1,0,0,0,0,0,0,1,0,1,0,1,0,0,0,1,0,0,0,0,0,0,5.000000e-01,0.866025,0.201299,0.979530,0,1,0,0,0,0,0,1,0,0
242949,0.454056,0.000000,-0.570892,-0.592211,0.0,-0.601360,0.0,-0.580821,-0.517768,0.748422,0.000000,0.000000,0.000000,0.0,0.044619,-0.220216,0.0,0.0,0.0,1.0,1.0,0,0.000000,0.000000,0.0,0.000000,0.0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,1,1,1,0,0,1,0,1,0,1,0,0,0,1,1,1,0,0,0,0,1,0,0,1,1,0,0,0,0,0,1,0,1,0,1,0,1,0,1,0,0,1,0,0,0,0,0,0,0,-8.660254e-01,-0.500000,0.998717,-0.050649,0,1,0,0,0,0,0,1,0,1
130533,0.210425,1.182658,0.409325,0.424611,0.0,0.728207,0.0,0.518443,0.082914,-0.193984,0.862406,1.062626,1.049584,0.0,0.904329,0.412438,0.0,0.0,1.0,0.0,0.0,0,0.822864,0.000000,0.0,0.000000,0.0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,0,1,1,1,0,1,1,0,1,0,1,1,0,0,1,0,1,0,0,1,1,1,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,-5.000000e-01,0.866025,0.790776,-0.612106,0,1,0,0,0,0,1,0,0,0
287524,-0.650190,0.000000,-0.570892,-0.592211,0.0,-0.601360,0.0,-0.580821,0.712169,-0.586678,0.000000,0.000000,0.000000,0.0,0.044619,-0.220216,0.0,0.0,0.0,1.0,0.0,0,0.000000,0.000000,0.0,0.000000,0.0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,0,1,1,1,0,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,1,0,0,1,0,1,0,1,0,0,0,1,0,0,0,0,0,0,1.224647e-16,-1.000000,0.937752,0.347305,0,1,0,0,1,0,0,0,0,1


In [53]:
x_val = pd.read_parquet("../data_processed/x_val.parquet")
x_val.head()

,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Bwd Packet Length Max,Bwd Packet Length Min,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Bwd IAT Total,Bwd IAT Mean,Fwd PSH Flags,Fwd Header Length,Bwd Packets/s,Min Packet Length,FIN Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,Down/Up Ratio,Init_Win_bytes_backward,Active Mean,Active Std,Idle Mean,Idle Std,s_bit_0,s_bit_1,s_bit_2,s_bit_3,s_bit_4,s_bit_5,s_bit_6,s_bit_7,s_bit_8,s_bit_9,s_bit_10,s_bit_11,s_bit_12,s_bit_13,s_bit_14,s_bit_15,s_bit_16,s_bit_17,s_bit_18,s_bit_19,s_bit_20,s_bit_21,s_bit_22,s_bit_23,s_bit_24,s_bit_25,s_bit_26,s_bit_27,s_bit_28,s_bit_29,s_bit_30,s_bit_31,d_bit_0,d_bit_1,d_bit_2,d_bit_3,d_bit_4,d_bit_5,d_bit_6,d_bit_7,d_bit_8,d_bit_9,d_bit_10,d_bit_11,d_bit_12,d_bit_13,d_bit_14,d_bit_15,d_bit_16,d_bit_17,d_bit_18,d_bit_19,d_bit_20,d_bit_21,d_bit_22,d_bit_23,d_bit_24,d_bit_25,d_bit_26,d_bit_27,d_bit_28,d_bit_29,d_bit_30,d_bit_31,Source Port_Grp_System,Source Port_Grp_Registered,Source Port_Grp_Dynamic,Destination Port_Grp_System,Destination Port_Grp_Registered,Destination Port_Grp_Dynamic,Destination Port_Flag_HTTP,Destination Port_Flag_HTTPS,Destination Port_Flag_DNS,Destination Port_Flag_Reserved,Destination Port_Flag_SSDP,Destination Port_Flag_Other_System,Destination Port_Flag_Other_Registered,Destination Port_Flag_Dynamic,Hour_sin,Hour_cos,Day_sin,Day_cos,Protocol_0.0,Protocol_6.0,Protocol_17.0,DayOfWeek_0,DayOfWeek_1,DayOfWeek_2,DayOfWeek_3,DayOfWeek_4,DayOfWeek_5,Init_Win_bytes_Bwd_Missing
108563,0.308921,0.000000,-0.570892,-0.592211,0.0,-0.601360,0.0,-0.580821,-0.450671,0.572945,0.000000,0.000000,0.000000,0.0,-0.279107,-0.220216,0.0,0.0,0.0,1.0,1.0,0,0.000000,0.000000,0.0,0.000000,0.0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,1,1,0,1,1,1,1,1,1,0,1,0,0,0,1,1,1,0,0,1,1,1,1,0,1,0,1,1,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0.500000,-0.866025,0.848644,0.528964,0,1,0,0,0,0,1,0,0,1
339692,0.639939,0.261860,-0.570892,-0.592211,0.0,-0.601360,0.0,-0.580821,-0.535380,0.726925,1.131617,0.000000,0.000000,0.0,0.199214,-0.201096,0.0,0.0,1.0,0.0,0.0,0,1.636899,0.000000,0.0,0.000000,0.0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,0,0,1,1,1,0,1,0,0,0,1,1,1,1,1,1,1,1,1,0,1,1,0,0,1,1,0,1,0,1,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,-0.866025,0.500000,0.897805,-0.440394,0,1,0,0,0,0,0,1,0,0
145288,-0.146969,0.000000,-0.570892,-0.592211,0.0,-0.601360,0.0,-0.580821,0.103439,-0.133614,0.841605,0.000000,0.000000,0.0,0.044619,0.335725,0.0,0.0,0.0,1.0,1.0,0,1.011778,0.000000,0.0,0.000000,0.0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,1,0,1,1,0,0,1,1,0,1,1,0,0,1,0,0,0,0,1,0,1,0,1,1,1,0,1,0,1,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0.866025,-0.500000,0.724793,0.688967,0,1,0,0,0,0,1,0,0,0
74532,0.866925,0.261860,-0.570892,-0.592211,0.0,0.031000,0.0,-0.527482,-0.569108,0.936883,1.208750,1.435116,1.681092,0.0,0.126160,-0.208849,0.0,0.0,0.0,1.0,0.0,0,1.136141,0.000000,0.0,0.000000,0.0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,1,1,1,0,1,1,0,0,1,1,1,1,1,1,0,1,0,1,1,0,0,1,0,1,1,1,0,1,1,0,1,0,0,1,1,0,1,0,1,0,0,0,1,0,0,0,0,0,0,-0.866025,-0.500000,0.998717,-0.050649,0,1,0,0,0,1,0,0,0,0
87030,0.755895,1.929947,0.804863,0.652002,0.0,0.728207,0.0,0.188997,-0.343570,0.245678,1.072755,1.386679,1.326063,0.0,1.573624,-0.024373,0.0,0.0,1.0,0.0,0.0,1,1.128043,14.956261,0.0,17.216063,0.0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,1,1,1,1,0,1,0,0,0,1,1,1,0,1,1,0,0,0,1,1,0,0,1,0,1,1,1,1,0,1,0,0,1,1,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0.866025,0.500000,0.394356,0.918958,0,1,0,0,0,0,1,0,0,0


In [54]:
y_train = pd.read_parquet("../data_processed/y_train.parquet")
y_train.head()

,Label
341993,3
300839,2
242949,1
130533,0
287524,2


In [55]:
y_val = pd.read_parquet("../data_processed/y_val.parquet")
y_val.head()

,Label
108563,0
339692,3
145288,0
74532,0
87030,0


## **3. Thực hiện vấn đề**

### 3.1. Oversample class Benign và tách tập Label thành 2 loại Benign và Malware

In [ ]:
y_train_bin = (y_train != 3).astype(int)
y_val_bin  = (y_val != 3).astype(int)

### 3.2. Lấy ra các dòng trong tập train là Benign

In [ ]:
y_train_np = y_train.values
x_train_normal = x_train[y_train_np == 3]

### 3.3. Thiết lập các mô hình bán giám sát

In [58]:
class AutoEncoderDetector:
    def __init__(self, model, threshold=None):
        self.model = model
        self.threshold = threshold

    def fit(self, X_train):
        self.model.fit(
            X_train, X_train,
            epochs=50,
            batch_size=256,
            verbose=0
        )

        recon = self.model.predict(X_train, verbose=0)
        error = np.mean((recon - X_train) ** 2, axis=1)

        self.threshold = np.percentile(error, 70)

    def predict(self, X):
        recon = self.model.predict(X)
        error = np.mean((recon - X) ** 2, axis=1)

        return (error > self.threshold).astype(int)

    def decision_function(self, X):
        recon = self.model.predict(X)
        error = np.mean((recon - X) ** 2, axis=1)
        return error

In [59]:
def build_autoencoder(input_dim):
    model = Sequential([
        Dense(128, activation='relu', input_shape=(input_dim,)),
        Dense(64, activation='relu'),
        Dense(32, activation='relu'),   # bottleneck
        Dense(64, activation='relu'),
        Dense(128, activation='relu'),
        Dense(input_dim, activation='linear')
    ])

    model.compile(optimizer='adam', loss='mse')
    return model

In [60]:
models = {
    "isolation_forest": IsolationForest(
        n_estimators=200,
        contamination=0.05,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "oneclass_svm": OneClassSVM(
        kernel="rbf",
        nu=0.2
    ),

    "lof": LocalOutlierFactor(
        n_neighbors=20,
        contamination=0.05,
        novelty=True
    ),
    "elliptic_envelope": EllipticEnvelope(
        contamination=0.05,
        random_state=RANDOM_STATE
    ),
}

ae_model = build_autoencoder(x_train.shape[1])
models["auto_encoder"] = AutoEncoderDetector(ae_model)

d:\Installs\Anaconda\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### 3.4. Thực hiện train và đánh giá kết quả

In [ ]:
for name, model in models.items():
    print(f"Running {name}...")

    model.fit(x_train_normal)

    y_pred = model.predict(x_val)
    y_pred = (y_pred == -1).astype(int)
    
    print(classification_report(y_val_bin, y_pred))
    print("ROC-AUC:", roc_auc_score(y_val_bin, y_pred))
    print(confusion_matrix(y_val_bin, y_pred))

Running isolation_forest...
              precision    recall  f1-score   support

           0       0.07      0.95      0.13      2371
           1       0.95      0.06      0.12     33192

    accuracy                           0.12     35563
   macro avg       0.51      0.51      0.12     35563
weighted avg       0.89      0.12      0.12     35563

ROC-AUC: 0.5058764471354324
[[ 2250   121]
 [31108  2084]]
Running oneclass_svm...


**Nhận xét:** Mô hình LOF hoạt động tốt nhất với ROC-AUC score là 0.64

# **Kết thúc**